In [2]:
import numpy as np
from StatTools.generators.ndfnoise_generator import ndfnoise
from StatTools.generators.multi_scale_fractional_generator import MultiScaleFractionalGenerator
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
 
def gen_poisson_bg(frame_shape, num_photons= 1000, qe = 0.6, sense = 0.36, bitdepth = 8):
    max_adu = int(2**bitdepth - 1)
 
    mu_p = np.ones(frame_shape) * num_photons
    poisson = np.random.poisson(lam=mu_p, size=frame_shape)
    electrons = np.round(poisson * qe)
    adu = (electrons * sense).astype(np.int32)
    adu[adu > max_adu] = max_adu
    return adu.astype(np.uint8)

def gen_traj(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
             hurst_move: float = 0.5, hurst_species: float = 0.5, 
             start_point = None):
    

    dx = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)
    dy = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(margin, frame_shape[1] - margin, (ants_num,))
        start_y = np.random.randint(margin, frame_shape[0] - margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

def gen_traj_corr(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
                    h_list: list=None, crossover_points:list=None,
                    start_point = None, correlation_matrix: np.ndarray = None):
    
    if correlation_matrix is None:
        _r = np.random.rand(ants_num, ants_num)
        cov = _r @ _r.T
        var = cov.diagonal()
        d = np.sqrt(var)
        correlation_matrix = cov / np.outer(d,d)
        np.fill_diagonal(correlation_matrix, 1.0)
        
    generator = MultiScaleFractionalGenerator(h_list=h_list, crossover_points=crossover_points)
    dx = generator.generate(frame_num, ants_num, correlation_matrix=correlation_matrix).T
    dy = generator.generate(frame_num, ants_num, correlation_matrix=correlation_matrix).T

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(-margin, frame_shape[1] + margin, (ants_num,))
        start_y = np.random.randint(-margin, frame_shape[0] + margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

def draw_traj(trajectories, frame_shape: tuple, ants_num: int):
    
    colors = sns.color_palette(palette='bright', n_colors=ants_num)
    colors = [tuple([int(c*255) for c in color]) for color in colors]

    bg = np.zeros(shape=(*frame_shape, 3), dtype=np.uint8)
    trajs = []
    for ant in range(trajectories.shape[1]):
        trajs.append(np.array([np.array(traj) for traj in trajectories[:, ant]]))
    for ant, color in  zip(trajs, colors):

        cv2.polylines(bg, [ant], isClosed=False, color=color, thickness=2)

    plt.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB))
    plt.show()

def draw_ants_with_direction(trajectories, output_path:str, frame_num: int, ants_num: int, frame_shape: tuple, ant_length = 5, ant_width=2, smooth_window:int=3, imshow:bool=False):
    
    
    # Define codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    frame_size = (frame_shape[1], frame_shape[0]) 
    out = cv2.VideoWriter(output_path, fourcc, 24.0, frame_size, isColor=True)
    for frame_idx in range(frame_num):
        frame = gen_poisson_bg(frame_shape, sense=0.43)
        # frame = np.ones((*frame_shape, 3), dtype=np.uint8)*255
        
        for ant_idx in range(ants_num):
            
            x, y = trajectories[frame_idx, ant_idx]
            start_index = max(0, frame_idx-smooth_window)
            dx = x - trajectories[start_index:frame_idx, ant_idx, 0].mean()
            dy = y - trajectories[start_index:frame_idx, ant_idx, 1].mean()
            angle = np.arctan2(dy, dx) * 180 / np.pi
            
            center = (int(x), int(y))
            axes = (ant_length, ant_width)
            cv2.ellipse(frame, center, axes, angle, 0, 360, 
                       (100, 100, 100), -1)
        frame = cv2.GaussianBlur(frame, ksize=(7,7), sigmaX=1.0)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        out.write(frame)
        if imshow:
            cv2.imshow('frame', frame)
            if cv2.waitKey(1) & 0xFF == 27:
                cv2.destroyAllWindows()
                out.release()
                break
        
    out.release()
    cv2.destroyAllWindows()

In [3]:
# config = {
# 'frame_num' : 10000,
# 'frame_shape' : (720, 1280),
# 'margin': 200,
# 'hurst_move': 1.0,
# 'hurst_species': 0.5,
# 'ants_num' : 200,
# 'start_point' : None
# }
# trajectories = gen_traj(**config)

# out_of_bounds = ((trajectories <= config['frame_shape'][::-1]).all(axis=-1)) & ((trajectories >= 0).all(axis=-1))
# out_of_bounds = np.expand_dims(out_of_bounds, axis=2).repeat(2, axis=2)

# trajectories_cut = np.where(out_of_bounds, trajectories, np.nan)
# draw_ants_with_direction(trajectories,
#                             output_path = f'data/gen_ants.mp4',
#                             frame_num=config['frame_num'],
#                             ants_num=config['ants_num'],
#                             frame_shape=config['frame_shape'],
#                             ant_length = 3,
#                             ant_width=1,
#                             smooth_window=5)

# flat_trajectories = trajectories_cut.reshape(-1, 2)

# frame_indices = np.repeat(np.arange(trajectories_cut.shape[0]), trajectories_cut.shape[1])
# ant_indices = np.tile(np.arange(trajectories_cut.shape[1]), trajectories_cut.shape[0])

# df = pd.DataFrame({
#     'frame': frame_indices,
#     'ant_id': ant_indices,
#     'x': flat_trajectories[:, 0],
#     'y': flat_trajectories[:, 1]
# })
# df = df.sort_values(['ant_id','frame'])
# df.to_csv('data/gen_ants_gt.csv', index=False)

In [4]:
np.random.seed(42)
for idx in tqdm(range(20)):
    config = {
    'frame_num' : 10000,
    'frame_shape' : (720, 1280),
    'margin': 200,
    'hurst_move': 1.0,
    'hurst_species': 0.5,
    'ants_num' : 100,
    'start_point' : None
    }
    trajectories = gen_traj(**config)

    out_of_bounds = ((trajectories <= config['frame_shape'][::-1]).all(axis=-1)) & ((trajectories >= 0).all(axis=-1))
    out_of_bounds = np.expand_dims(out_of_bounds, axis=2).repeat(2, axis=2)

    trajectories_cut = np.where(out_of_bounds, trajectories, np.nan)
    draw_ants_with_direction(trajectories,
                             output_path = f'data/gen_ants_{idx}.mp4',
                             frame_num=config['frame_num'],
                             ants_num=config['ants_num'],
                             frame_shape=config['frame_shape'],
                             ant_length = 3,
                             ant_width=1,
                             smooth_window=5)
    
    flat_trajectories = trajectories_cut.reshape(-1, 2)

    frame_indices = np.repeat(np.arange(trajectories_cut.shape[0]), trajectories_cut.shape[1])
    ant_indices = np.tile(np.arange(trajectories_cut.shape[1]), trajectories_cut.shape[0])

    df = pd.DataFrame({
        'frame': frame_indices,
        'ant_id': ant_indices,
        'x': flat_trajectories[:, 0],
        'y': flat_trajectories[:, 1]
    })
    df = df.sort_values(['ant_id','frame'])
    df.to_csv(f'data/gen_ants_gt_{idx}.csv', index=False)
    

  0%|          | 0/20 [00:00<?, ?it/s]

/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
/tmp/ipykernel_222004/1635430116.py:102: RuntimeWarning: Mean of empty slice.
  dx = x - trajectories[start_index:frame_idx, ant_idx, 0].mean()
/home/akhiyarov/.local/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_222004/1635430116.py:103: RuntimeWarning: Mean of empty slice.
  dy = y - trajectories[start_index:frame_idx, ant_idx, 1].mean()
  5%|▌         | 1/20 [05:02<1:35:49, 302.60s/it]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
/tmp/ipykernel_222004/1635430116.py:102: RuntimeWarning: Mean of empty

In [5]:
# frame_shape = (500,500)
# frame_num=4000
# ants_num=50
# trajectories_msfg = gen_traj_corr(frame_num=4000, ants_num=50, frame_shape=(500,500), h_list=[1.0, 1.0], crossover_points=[1000])
# draw_traj(trajectories_msfg, frame_shape,  ants_num)

In [6]:
# flat_trajectories = trajectories.reshape(-1, 2)

# frame_indices = np.repeat(np.arange(trajectories.shape[0]), trajectories.shape[1])
# ant_indices = np.tile(np.arange(trajectories.shape[1]), trajectories.shape[0])

# df = pd.DataFrame({
#     'frame': frame_indices,
#     'ant_id': ant_indices,
#     'x': flat_trajectories[:, 0],
#     'y': flat_trajectories[:, 1]
# })
# df = df.sort_values(['ant_id','frame'])
# df.to_csv('gen_ants_gt.csv', index=False)

In [7]:
# draw_ants_with_direction(trajectories,
#                          output_path = 'gen_ants.mp4',
#                          frame_num=frame_num,
#                          ants_num=ants_num,
#                          frame_shape=frame_shape,
#                          ant_length = 3,
#                          ant_width=1,
#                          smooth_window=5)